In [11]:
import os
import cv2
import easyocr
from openpyxl import Workbook
from pathlib import Path
import warnings

INPUT_ROOT = "Datasets/PZ1"          
OUTPUT_ROOT = "outputs"             
EXCEL_FILE = "meter_results.xlsx"

SUPPORTED_EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif')

reader = easyocr.Reader(['en'], gpu=True)  

Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)

wb = Workbook()
ws = wb.active
ws.title = "Results"
ws.append(["Исходный путь (относительный)", "Номер счётчика (6 цифр)", "Путь к бинарному изображению"])


def process_image(image_path: str) -> tuple:
    """
    Загружает изображение, бинаризует, распознаёт текст,
    возвращает (номер_счётчика, бинарное_изображение_массив)
    номер_счётчика = None если не найден.
    """
    img = cv2.imread(image_path)
    if img is None:
        return None, None

    # Ч/б и бинаризация
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    binary = cv2.adaptiveThreshold(gray, 255,
                                   cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                   cv2.THRESH_BINARY_INV, 11, 2)

    # OCR
    results = reader.readtext(binary)
    meter_number = None
    for (bbox, text, prob) in results:
        digits = ''.join(filter(str.isdigit, text))
        if len(digits) == 6:
            meter_number = digits
            break

    return meter_number, binary


def get_output_path(original_path: str, root_input: str, root_output: str) -> str:
    rel_path = os.path.relpath(original_path, root_input)  
    parent_dir = os.path.dirname(rel_path)                 
    filename = os.path.basename(rel_path)                 
    name_without_ext = Path(filename).stem                 
    ext = Path(filename).suffix                            
    binary_filename = f"binary_{name_without_ext}{ext}"    

    out_dir = os.path.join(root_output, parent_dir)
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    return os.path.join(out_dir, binary_filename)


def main():
    if not os.path.isdir(INPUT_ROOT):
        print(f"Ошибка: корневая папка '{INPUT_ROOT}' не существует.")
        return

    image_files = []
    for root, dirs, files in os.walk(INPUT_ROOT):
        for file in files:
            if file.lower().endswith(SUPPORTED_EXT):
                full_path = os.path.join(root, file)
                image_files.append(full_path)

    if not image_files:
        print(f"В папке '{INPUT_ROOT}' нет изображений.")
        return

    print(f"Найдено изображений: {len(image_files)}")
    print("Обработка... (может занять время, используется GPU)")

    for i, img_path in enumerate(image_files, 1):
        # Относительный путь для отображения в консоли и Excel
        rel_path = os.path.relpath(img_path, start=INPUT_ROOT)
        print(f"  [{i}/{len(image_files)}] {rel_path}")

        meter_number, binary_img = process_image(img_path)

        if binary_img is not None:
            out_path = get_output_path(img_path, INPUT_ROOT, OUTPUT_ROOT)
            cv2.imwrite(out_path, binary_img)
        else:
            out_path = "не сохранено (ошибка чтения)"

        result_str = meter_number if meter_number else "Не найден"
        ws.append([rel_path, result_str, out_path])

    wb.save(EXCEL_FILE)
    print(f"\nГотово! Результаты в '{EXCEL_FILE}'")
    print(f"Бинарные изображения сохранены в папке '{OUTPUT_ROOT}'")


if __name__ == "__main__":
    main()

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


Найдено изображений: 120
Обработка... (может занять время, используется GPU)
  [1/120] 1-ф сплит РОТЕК\390255.png
  [2/120] 1-ф сплит РОТЕК\390818.png
  [3/120] 1-ф сплит РОТЕК\390819.png
  [4/120] 1-ф сплит РОТЕК\390823.png
  [5/120] 1-ф сплит РОТЕК\390824.png
  [6/120] 1-ф сплит РОТЕК\392984.png
  [7/120] 1-ф сплит РОТЕК\392995.png
  [8/120] 1-ф сплит РОТЕК\399891.png
  [9/120] 1-ф сплит РОТЕК\404940.png
  [10/120] 1-ф сплит РОТЕК\404947.png
  [11/120] 1-ф сплит РОТЕК\405771.png
  [12/120] 1-ф сплит РОТЕК\407793.png
  [13/120] 1-ф сплит РОТЕК\410275.png
  [14/120] 1-ф сплит РОТЕК\410277.png
  [15/120] 1-ф сплит РОТЕК\415377.png
  [16/120] 1-ф сплит РОТЕК\419607.png
  [17/120] 1-ф сплит РОТЕК\419608.png
  [18/120] 1-ф сплит РОТЕК\419609.png
  [19/120] 1-ф сплит РОТЕК\419610.png
  [20/120] 1-ф сплит РОТЕК\419611.png
  [21/120] 1-ф сплит РОТЕК\419612.png
  [22/120] 1-ф сплит РОТЕК\419613.png
  [23/120] 1-ф сплит РОТЕК\419614.png
  [24/120] 1-ф сплит РОТЕК\419615.png
  [25/120] 1-ф сплит

In [35]:
import os
import sys
import tempfile
import shutil
import cv2
import yt_dlp

def get_video_url(url, quality='best'):
    """Получить прямую ссылку на видеопоток (не используется, оставлен для справки)."""
    ydl_opts = {
        'quiet': True,
        'format': quality,
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=False)
        if 'entries' in info:
            info = info['entries'][0]
        return info['url']

def download_video(url, output_path, quality='best'):
    """Скачать видео по ссылке в указанный файл."""
    ydl_opts = {
        'quiet': True,
        'format': quality,
        'outtmpl': output_path,
    }
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([url])
        return True
    except Exception as e:
        print(f"Ошибка скачивания: {e}")
        return False

def extract_frames(video_path, output_folder, fps=1):
    """Извлечь кадры из видеофайла с заданной частотой."""
    os.makedirs(output_folder, exist_ok=True)
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Не удалось открыть видео: {video_path}")
        return 0

    video_fps = cap.get(cv2.CAP_PROP_FPS)
    if video_fps <= 0:
        video_fps = 30.0

    frame_interval = max(1, int(round(video_fps / fps)))
    frame_count = 0
    saved_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_count % frame_interval == 0:
            out_path = os.path.join(output_folder, f"frame_{saved_count:06d}.jpg")
            cv2.imwrite(out_path, frame)
            saved_count += 1
        frame_count += 1

    cap.release()
    print(f"Сохранено {saved_count} кадров в {output_folder}")
    return saved_count

def process_local_video(video_path, output_base_folder, fps):
    """Обработать одно локальное видео."""
    video_name = os.path.splitext(os.path.basename(video_path))[0]
    output_folder = os.path.join(output_base_folder, video_name)
    extract_frames(video_path, output_folder, fps)

def process_local_folder(folder_path, output_base_folder, fps):
    """Обработать все видео в локальной папке."""
    video_extensions = ('.mp4', '.avi', '.mkv', '.webm', '.mov', '.flv')
    for file in os.listdir(folder_path):
        if file.lower().endswith(video_extensions):
            video_path = os.path.join(folder_path, file)
            print(f"Обработка: {video_path}")
            process_local_video(video_path, output_base_folder, fps)

def process_url(url, output_base_folder, fps, quality='best'):
    """Обработать видео по URL (скачать во временную папку, затем нарезать)."""
    temp_dir = tempfile.mkdtemp()
    temp_video_path = os.path.join(temp_dir, 'video.mp4')
    print("Скачивание видео...")
    if download_video(url, temp_video_path, quality):
        output_folder = os.path.join(output_base_folder, 'downloaded_video')
        extract_frames(temp_video_path, output_folder, fps)
    else:
        print("Не удалось скачать видео.")
    shutil.rmtree(temp_dir, ignore_errors=True)

def main():
    print("=== Скрипт нарезки видео на кадры ===")
    mode = input("Выберите режим:\n1 - Ссылка на видео (Rutube, YouTube, Яндекс.Диск и др.)\n2 - Папка с видео на компьютере\nВаш выбор: ")

    fps = float(input("Введите частоту кадров (кадров/сек): "))
    if fps <= 0:
        fps = 1

    quality = input("Качество скачивания (по умолчанию 'best'): ").strip()
    if not quality:
        quality = 'best'

    output_base = input("Базовая папка для кадров (по умолчанию 'frames'): ").strip()
    if not output_base:
        output_base = 'frames'

    if mode == '1':
        url = input("Введите ссылку на видео: ").strip()
        process_url(url, output_base, fps, quality)
    elif mode == '2':
        folder_path = input("Путь к папке с видео: ").strip()
        if not os.path.isdir(folder_path):
            print("Папка не найдена.")
            return
        process_local_folder(folder_path, output_base, fps)
    else:
        print("Неверный режим.")

if __name__ == '__main__':
    main()

=== Скрипт нарезки видео на кадры ===
Скачивание видео...


Сохранено 891 кадров в pp\downloaded_video                 


In [21]:
import os
import cv2
import easyocr
from openpyxl import Workbook
from pathlib import Path
import warnings

warnings.filterwarnings("ignore", category=UserWarning)  # отключаем лишние предупреждения

# ----------------------------- НАСТРОЙКИ -----------------------------
INPUT_FOLDER = "frames"          # папка с кадрами (можно изменить)
OUTPUT_EXCEL = "recognized_text.xlsx"
OCR_LANGUAGES = ['ru', 'en']     # русский + английский
CROP_BOTTOM_RATIO = 0.2          # обрезать нижние 20% (зона субтитров), 0 = не обрезать
MIN_TEXT_LENGTH = 2              # минимальная длина текста для сохранения

# ----------------------------- ИНИЦИАЛИЗАЦИЯ OCR ---------------------
print("Загрузка EasyOCR (первый запуск может быть долгим)...")
reader = easyocr.Reader(OCR_LANGUAGES, gpu=False)  # gpu=True если есть видеокарта
print("OCR готов.")

def extract_text_from_image(image_path):
    """
    Загружает изображение, опционально обрезает нижнюю часть,
    распознаёт текст с помощью EasyOCR.
    Возвращает строку с распознанным текстом (все слова через пробел).
    """
    # читаем изображение
    img = cv2.imread(image_path)
    if img is None:
        return ""

    # обрезаем нижнюю часть, если задано
    if 0 < CROP_BOTTOM_RATIO < 1:
        h, w = img.shape[:2]
        crop_y = int(h * (1 - CROP_BOTTOM_RATIO))
        img = img[crop_y:h, :]

    # распознавание
    results = reader.readtext(img, detail=0, paragraph=True)
    # results – список строк (каждый абзац или блок текста)
    full_text = " ".join(results).strip()
    return full_text

def process_folder():
    """Обходит папку INPUT_FOLDER, обрабатывает все изображения и сохраняет результат в Excel."""
    folder_path = Path(INPUT_FOLDER)
    if not folder_path.exists():
        print(f"Папка {INPUT_FOLDER} не найдена!")
        return

    # собираем все изображения
    image_files = list(folder_path.rglob("*.jpg")) + list(folder_path.rglob("*.png"))
    image_files.sort()
    print(f"Найдено {len(image_files)} кадров. Начинаем обработку...")

    wb = Workbook()
    ws = wb.active
    ws.title = "Распознанный текст"
    ws.append(["Файл", "Распознанный текст"])

    for idx, img_path in enumerate(image_files, 1):
        rel_path = img_path.relative_to(folder_path)
        print(f"[{idx}/{len(image_files)}] Обработка {rel_path}")

        text = extract_text_from_image(str(img_path))
        if len(text) >= MIN_TEXT_LENGTH:
            ws.append([str(rel_path), text])
        else:
            ws.append([str(rel_path), ""])  # пустая строка, если текст короткий

    # сохраняем Excel
    wb.save(OUTPUT_EXCEL)
    print(f"\nГотово! Результат сохранён в {OUTPUT_EXCEL}")

if __name__ == "__main__":
    # можно ввести параметры с клавиатуры для удобства
    inp = input(f"Введите путь к папке с кадрами (Enter для '{INPUT_FOLDER}'): ").strip()
    if inp:
        INPUT_FOLDER = inp
    process_folder()

Using CPU. Note: This module is much faster with a GPU.


Загрузка EasyOCR (первый запуск может быть долгим)...
Progress: |██████████████████████████████████████████████████| 100.1% CompleteOCR готов.
Найдено 1210 кадров. Начинаем обработку...
[1/1210] Обработка downloaded_video\frame_000000.jpg
[2/1210] Обработка downloaded_video\frame_000001.jpg
[3/1210] Обработка downloaded_video\frame_000002.jpg
[4/1210] Обработка downloaded_video\frame_000003.jpg
[5/1210] Обработка downloaded_video\frame_000004.jpg
[6/1210] Обработка downloaded_video\frame_000005.jpg
[7/1210] Обработка downloaded_video\frame_000006.jpg
[8/1210] Обработка downloaded_video\frame_000007.jpg
[9/1210] Обработка downloaded_video\frame_000008.jpg
[10/1210] Обработка downloaded_video\frame_000009.jpg
[11/1210] Обработка downloaded_video\frame_000010.jpg
[12/1210] Обработка downloaded_video\frame_000011.jpg
[13/1210] Обработка downloaded_video\frame_000012.jpg
[14/1210] Обработка downloaded_video\frame_000013.jpg
[15/1210] Обработка downloaded_video\frame_000014.jpg
[16/1210] Обр

In [36]:
import os
import cv2
import easyocr
from openpyxl import Workbook, load_workbook
from pathlib import Path
import warnings
from difflib import SequenceMatcher

warnings.filterwarnings("ignore", category=UserWarning)

# ----------------------------- НАСТРОЙКИ -----------------------------
INPUT_FOLDER = "pp"               # папка с кадрами
OUTPUT_EXCEL = "recognized_textpp.xlsx"
OUTPUT_TXT = "recognized_textpp.txt"
OCR_LANGUAGES = ['en']          
CROP_BOTTOM_RATIO = 0.2               # обрезать нижние 20% (субтитры)
MIN_TEXT_LENGTH = 2                   # минимальная длина текста для сохранения
SIMILARITY_THRESHOLD = 0.85           # порог схожести (0..1) для пропуска дублей

def is_duplicate(new_text, last_text, threshold=SIMILARITY_THRESHOLD):
    """Проверяет, является ли новый текст дубликатом предыдущего."""
    if not last_text:
        return False
    # Приводим к нижнему регистру и удаляем лишние пробелы
    new_clean = " ".join(new_text.lower().split())
    last_clean = " ".join(last_text.lower().split())
    if new_clean == last_clean:
        return True
    # Вычисляем процент схожести
    ratio = SequenceMatcher(None, new_clean, last_clean).ratio()
    return ratio >= threshold

def extract_text_from_image(image_path):
    """Распознаёт текст из нижней части изображения."""
    img = cv2.imread(image_path)
    if img is None:
        return ""

    # Обрезаем нижнюю часть, если задано
    if 0 < CROP_BOTTOM_RATIO < 1:
        h, w = img.shape[:2]
        crop_y = int(h * (1 - CROP_BOTTOM_RATIO))
        img = img[crop_y:h, :]

    results = reader.readtext(img, detail=0, paragraph=True)
    full_text = " ".join(results).strip()
    return full_text

def process_frames_to_excel():
    """Обходит папку, распознаёт текст, отфильтровывает дубликаты, записывает в Excel."""
    folder_path = Path(INPUT_FOLDER)
    if not folder_path.exists():
        print(f"Папка {INPUT_FOLDER} не найдена!")
        return False

    image_files = list(folder_path.rglob("*.jpg")) + list(folder_path.rglob("*.png"))
    image_files.sort()
    print(f"Найдено {len(image_files)} кадров. Распознавание...")

    wb = Workbook()
    ws = wb.active
    ws.title = "Распознанный текст"
    ws.append(["Файл", "Распознанный текст"])

    last_text = ""  # последний добавленный текст (не пустой)
    added_count = 0

    for idx, img_path in enumerate(image_files, 1):
        rel_path = img_path.relative_to(folder_path)
        print(f"[{idx}/{len(image_files)}] {rel_path}")

        text = extract_text_from_image(str(img_path))

        # Пропускаем короткий текст
        if len(text) < MIN_TEXT_LENGTH:
            continue

        # Проверка на дубликат с предыдущим
        if is_duplicate(text, last_text):
            print(f"  → Дубликат, пропущен: {text[:50]}...")
            continue

        # Добавляем в Excel
        ws.append([str(rel_path), text])
        last_text = text
        added_count += 1
        print(f"  → Добавлен: {text[:60]}...")

    wb.save(OUTPUT_EXCEL)
    print(f"\nExcel сохранён: {OUTPUT_EXCEL} (добавлено строк: {added_count})")
    return True

def export_excel_to_txt():
    """Читает Excel-файл и выписывает все тексты в TXT (уже без дублей)."""
    if not Path(OUTPUT_EXCEL).exists():
        print(f"Файл {OUTPUT_EXCEL} не найден. Сначала выполните распознавание.")
        return

    wb = load_workbook(OUTPUT_EXCEL)
    ws = wb.active

    texts = []
    for row in ws.iter_rows(min_row=2, values_only=True):  # пропускаем заголовок
        file_name, recognized_text = row
        if recognized_text and len(recognized_text.strip()) >= MIN_TEXT_LENGTH:
            texts.append(recognized_text.strip())

    with open(OUTPUT_TXT, "w", encoding="utf-8") as f:
        f.write("\n".join(texts))

    print(f"TXT сохранён: {OUTPUT_TXT} (всего строк: {len(texts)})")

# ----------------------------- ОСНОВНАЯ ЧАСТЬ -----------------------------
if __name__ == "__main__":
    print("Загрузка EasyOCR...")
    reader = easyocr.Reader(OCR_LANGUAGES, gpu=False)
    print("OCR готов.\n")

    # Шаг 1: распознавание и создание Excel (с удалением дублей)
    if process_frames_to_excel():
        # Шаг 2: из Excel в TXT
        export_excel_to_txt()
    else:
        print("Не удалось обработать кадры.")

Using CPU. Note: This module is much faster with a GPU.


Загрузка EasyOCR...
OCR готов.

Найдено 891 кадров. Распознавание...
[1/891] downloaded_video\frame_000000.jpg
[2/891] downloaded_video\frame_000001.jpg
[3/891] downloaded_video\frame_000002.jpg
[4/891] downloaded_video\frame_000003.jpg
[5/891] downloaded_video\frame_000004.jpg
[6/891] downloaded_video\frame_000005.jpg
[7/891] downloaded_video\frame_000006.jpg
[8/891] downloaded_video\frame_000007.jpg
[9/891] downloaded_video\frame_000008.jpg
[10/891] downloaded_video\frame_000009.jpg
[11/891] downloaded_video\frame_000010.jpg
[12/891] downloaded_video\frame_000011.jpg
  → Добавлен: ean don ( wanna eave...
[13/891] downloaded_video\frame_000012.jpg
  → Дубликат, пропущен: ean don [ wanna eave IL...
[14/891] downloaded_video\frame_000013.jpg
  → Добавлен: Tean; don [ wanna eave IL...
[15/891] downloaded_video\frame_000014.jpg
  → Дубликат, пропущен: ean; aon [ wanna eave...
[16/891] downloaded_video\frame_000015.jpg
  → Дубликат, пропущен: Tean; | don ( wanna eave...
[17/891] downloaded

In [37]:
# PZ1: Распознавание 6-значного номера счётчика на изображениях
# Библиотеки: pip install opencv-python easyocr openpyxl

import os
import cv2
import easyocr
from openpyxl import Workbook
from pathlib import Path

# ===== НАСТРОЙКИ =====
INPUT_DIR = "Datasets/PZ1"        # папка с исходными изображениями
OUTPUT_DIR = "outputs"            # папка для бинарных копий
EXCEL_FILE = "meter_results.xlsx"

reader = easyocr.Reader(['en'], gpu=False)   # gpu=True если есть CUDA
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

wb = Workbook()
ws = wb.active
ws.append(["Относительный путь", "Номер счётчика (6 цифр)", "Путь к бинарному изображению"])

def process_image(path):
    img = cv2.imread(path)
    if img is None:
        return None, None
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    binary = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                   cv2.THRESH_BINARY_INV, 11, 2)
    results = reader.readtext(binary)
    for (_, text, _) in results:
        digits = ''.join(filter(str.isdigit, text))
        if len(digits) == 6:
            return digits, binary
    return None, binary

def get_output_path(orig, root_in, root_out):
    rel = os.path.relpath(orig, root_in)
    parent = os.path.dirname(rel)
    name = Path(rel).stem
    ext = Path(rel).suffix
    out_dir = os.path.join(root_out, parent)
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    return os.path.join(out_dir, f"binary_{name}{ext}")

# Сбор всех изображений
images = []
for root, _, files in os.walk(INPUT_DIR):
    for f in files:
        if f.lower().endswith(('.jpg','.jpeg','.png','.bmp','.tiff')):
            images.append(os.path.join(root, f))

print(f"Найдено изображений: {len(images)}")
for i, img_path in enumerate(images, 1):
    rel = os.path.relpath(img_path, INPUT_DIR)
    print(f"[{i}/{len(images)}] {rel}")
    number, binary = process_image(img_path)
    out_path = get_output_path(img_path, INPUT_DIR, OUTPUT_DIR) if binary is not None else "ошибка"
    if binary is not None:
        cv2.imwrite(out_path, binary)
    ws.append([rel, number if number else "Не найден", out_path])

wb.save(EXCEL_FILE)
print(f"Готово! Результат: {EXCEL_FILE}, бинарные изображения: {OUTPUT_DIR}")

Using CPU. Note: This module is much faster with a GPU.


Найдено изображений: 120
[1/120] 1-ф сплит РОТЕК\390255.png
[2/120] 1-ф сплит РОТЕК\390818.png
[3/120] 1-ф сплит РОТЕК\390819.png
[4/120] 1-ф сплит РОТЕК\390823.png
[5/120] 1-ф сплит РОТЕК\390824.png
[6/120] 1-ф сплит РОТЕК\392984.png
[7/120] 1-ф сплит РОТЕК\392995.png
[8/120] 1-ф сплит РОТЕК\399891.png
[9/120] 1-ф сплит РОТЕК\404940.png
[10/120] 1-ф сплит РОТЕК\404947.png
[11/120] 1-ф сплит РОТЕК\405771.png
[12/120] 1-ф сплит РОТЕК\407793.png
[13/120] 1-ф сплит РОТЕК\410275.png
[14/120] 1-ф сплит РОТЕК\410277.png
[15/120] 1-ф сплит РОТЕК\415377.png
[16/120] 1-ф сплит РОТЕК\419607.png
[17/120] 1-ф сплит РОТЕК\419608.png
[18/120] 1-ф сплит РОТЕК\419609.png
[19/120] 1-ф сплит РОТЕК\419610.png
[20/120] 1-ф сплит РОТЕК\419611.png
[21/120] 1-ф сплит РОТЕК\419612.png
[22/120] 1-ф сплит РОТЕК\419613.png
[23/120] 1-ф сплит РОТЕК\419614.png
[24/120] 1-ф сплит РОТЕК\419615.png
[25/120] 1-ф сплит РОТЕК\419616.png
[26/120] 1-ф сплит РОТЕК\419617.png
[27/120] 1-ф сплит РОТЕК\419619.png
[28/120] 1-ф

=== НАРЕЗКА ВИДЕО НА КАДРЫ (видео сохраняется в папку проекта) ===
Скачивание видео...
[youtube] Extracting URL: https://www.youtube.com/watch?v=GlunZpix30I
[youtube] GlunZpix30I: Downloading webpage


[youtube] GlunZpix30I: Downloading android vr player API JSON
[info] GlunZpix30I: Downloading 1 format(s): 18
[download] Destination: downloaded_videos\Learn English⧸Movies with English subtitles - Tony Stark  Spider-Man ｜ Spider-Man Homecoming.mp4
[download] 100% of    5.47MiB in 00:00:01 at 3.61MiB/s   
[youtube] Extracting URL: https://www.youtube.com/watch?v=GlunZpix30I
[youtube] GlunZpix30I: Downloading webpage


[youtube] GlunZpix30I: Downloading android vr player API JSON
[info] GlunZpix30I: Downloading 1 format(s): 18
[download] Destination: downloaded_videos\video_downloaded.mp4
[download] 100% of    5.47MiB in 00:00:00 at 7.90MiB/s   
Видео сохранено: downloaded_videos\video_downloaded.mp4
Сохранено 303 кадров в frames


In [ ]:
# PZ3: OCR субтитров из кадров + дедупликация
# Библиотеки: pip install opencv-python easyocr openpyxl

import os
import cv2
import easyocr
from openpyxl import Workbook
from pathlib import Path
from difflib import SequenceMatcher

# ===== НАСТРОЙКИ =====
FRAMES_DIR = "frames"                # папка с кадрами из PZ2
OUT_EXCEL = "subtitles.xlsx"
OUT_TXT = "subtitles.txt"
CROP_BOTTOM = 0.2                    # обрезать нижние 20% (зона субтитров)
SIMILARITY_THRESHOLD = 0.85

reader = easyocr.Reader(['en'], gpu=False)

def extract_text(img_path):
    img = cv2.imread(img_path)
    if img is None:
        return ""
    if 0 < CROP_BOTTOM < 1:
        h, w = img.shape[:2]
        img = img[int(h*(1-CROP_BOTTOM)):h, :]
    results = reader.readtext(img, detail=0, paragraph=True)
    return " ".join(results).strip()

def is_duplicate(new, last, thresh=SIMILARITY_THRESHOLD):
    if not last:
        return False
    new_clean = " ".join(new.lower().split())
    last_clean = " ".join(last.lower().split())
    if new_clean == last_clean:
        return True
    return SequenceMatcher(None, new_clean, last_clean).ratio() >= thresh

# Поиск кадров
frame_paths = list(Path(FRAMES_DIR).rglob("*.jpg")) + list(Path(FRAMES_DIR).rglob("*.png"))
frame_paths.sort()
print(f"Найдено кадров: {len(frame_paths)}")

wb = Workbook()
ws = wb.active
ws.append(["Файл", "Текст"])

last_text = ""
added = 0
for i, p in enumerate(frame_paths, 1):
    rel = p.relative_to(FRAMES_DIR)
    print(f"[{i}/{len(frame_paths)}] {rel}")
    text = extract_text(str(p))
    if len(text) < 2:
        continue
    if is_duplicate(text, last_text):
        print(f"  → дубль пропущен: {text[:50]}")
        continue
    ws.append([str(rel), text])
    last_text = text
    added += 1
    print(f"  → добавлен: {text[:60]}")

wb.save(OUT_EXCEL)
print(f"Excel сохранён: {OUT_EXCEL} (уникальных строк: {added})")

# Дополнительно сохраняем только текст в TXT
with open(OUT_TXT, "w", encoding="utf-8") as f:
    for row in ws.iter_rows(min_row=2, values_only=True):
        if row[1]:
            f.write(row[1].strip() + "\n")
print(f"Текстовый файл: {OUT_TXT}")

In [ ]:
# PZ4: Аудиодорожка из видео + транскрипция через Whisper
# Библиотеки: pip install opencv-python whisper openpyxl moviepy

import whisper
import moviepy.editor as mp
import os
from openpyxl import Workbook

# ===== НАСТРОЙКИ =====
VIDEO_FILE = "sample.mp4"          # путь к вашему видео
AUDIO_TEMP = "temp_audio.wav"
OUT_EXCEL = "speech_results.xlsx"

# 1. Извлечение звука из видео
def extract_audio(video_path, audio_path):
    video = mp.VideoFileClip(video_path)
    video.audio.write_audiofile(audio_path, verbose=False, logger=None)
    video.close()

# 2. Распознавание речи (Whisper)
model = whisper.load_model("base")   # tiny, base, small, medium, large
if not os.path.exists(AUDIO_TEMP):
    extract_audio(VIDEO_FILE, AUDIO_TEMP)

result = model.transcribe(AUDIO_TEMP, language="en")  # или "ru"
segments = result["segments"]

# 3. Сохранение в Excel
wb = Workbook()
ws = wb.active
ws.append(["Начало (сек)", "Конец (сек)", "Текст"])
for seg in segments:
    ws.append([round(seg['start'],2), round(seg['end'],2), seg['text'].strip()])

wb.save(OUT_EXCEL)
print(f"Транскрипция сохранена в {OUT_EXCEL}")

# Очистка временного файла (опционально)
# os.remove(AUDIO_TEMP)

In [ ]:
# PZ5: Обнаружение объектов на видео/изображениях с YOLOv8
# Библиотеки: pip install ultralytics opencv-python

import cv2
from ultralytics import YOLO
import os

# ===== НАСТРОЙКИ =====
INPUT_VIDEO = "sample.mp4"          # или папка с изображениями
OUTPUT_VIDEO = "yolo_output.mp4"
CONF_THRESH = 0.5

# Загрузка предобученной модели
model = YOLO("yolov8n.pt")          # n, s, m, l, x – разные размеры

# Обработка видео
cap = cv2.VideoCapture(INPUT_VIDEO)
fps = int(cap.get(cv2.CAP_PROP_FPS))
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
out = cv2.VideoWriter(OUTPUT_VIDEO, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

frame_idx = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    # Детекция
    results = model(frame, conf=CONF_THRESH)
    annotated = results[0].plot()   # отрисовка bbox и меток
    out.write(annotated)
    if frame_idx % 30 == 0:
        print(f"Обработано кадров: {frame_idx}")
    frame_idx += 1

cap.release()
out.release()
print(f"Готово! Результат: {OUTPUT_VIDEO}")

# Для одного изображения:
# img = cv2.imread("image.jpg")
# results = model(img)
# res_plotted = results[0].plot()
# cv2.imwrite("yolo_result.jpg", res_plotted)

In [ ]:
# PZ6: Классификация изображений с предобученным ResNet50 (ImageNet)
# Библиотеки: pip install torch torchvision opencv-python pillow

import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import cv2
import numpy as np

# Загрузка модели
model = models.resnet50(pretrained=True)
model.eval()

# Трансформации (размер 224x224, нормализация ImageNet)
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Загрузка меток классов ImageNet
LABELS_URL = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
import urllib.request
try:
    with urllib.request.urlopen(LABELS_URL) as f:
        classes = [line.decode().strip() for line in f.readlines()]
except:
    # fallback – локальный файл (если нет интернета)
    classes = [str(i) for i in range(1000)]

def classify_image(image_path):
    img = Image.open(image_path).convert('RGB')
    input_tensor = transform(img).unsqueeze(0)  # batch dimension
    with torch.no_grad():
        output = model(input_tensor)
    probabilities = torch.nn.functional.softmax(output[0], dim=0)
    top5_prob, top5_idx = torch.topk(probabilities, 5)
    results = [(classes[idx], prob.item()) for idx, prob in zip(top5_idx, top5_prob)]
    return results

# Пример использования
img_file = "test.jpg"   # замените на свой файл
preds = classify_image(img_file)
print("Топ-5 предсказаний:")
for label, prob in preds:
    print(f"  {label}: {prob*100:.2f}%")

In [ ]:
# PZ7: Анализ изображения через Gemini API (Google)
# Требуется API-ключ: https://aistudio.google.com/app/apikey
# Библиотеки: pip install google-generativeai pillow

import google.generativeai as genai
from PIL import Image
import os

# === НАСТРОЙКИ ===
GEMINI_API_KEY = "ВАШ_API_КЛЮЧ"    # получите бесплатно на Google AI Studio
genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel('gemini-1.5-flash')   # или 'gemini-1.5-pro'

def analyze_image(image_path, prompt="Опиши объекты на этом изображении."):
    img = Image.open(image_path)
    response = model.generate_content([prompt, img])
    return response.text

# Пример
img_path = "street.jpg"
description = analyze_image(img_path)
print("Ответ Gemini:\n", description)

# === ЗАМЕТКА О РАЗВЁРТЫВАНИИ НА TIMEWEB ===
# Для развёртывания на Timeweb (или любом хостинге) нужно:
# 1. Упаковать код в веб-приложение (Flask/FastAPI)
# 2. Создать endpoint, который принимает изображение и возвращает результат Gemini
# 3. Загрузить проект на сервер Timeweb (через git или FTP)
# 4. Установить зависимости и запустить через Gunicorn/uWSGI
#
# Ниже – минимальный пример Flask-приложения (сохраните как app.py):
"""
from flask import Flask, request, jsonify
import google.generativeai as genai
from PIL import Image
import io

app = Flask(__name__)
genai.configure(api_key=os.environ["GEMINI_KEY"])
model = genai.GenerativeModel('gemini-1.5-flash')

@app.route('/analyze', methods=['POST'])
def analyze():
    file = request.files['image']
    img = Image.open(file.stream)
    response = model.generate_content(["Опиши объекты на картинке", img])
    return jsonify({"result": response.text})

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000)
"""
# Запуск на Timeweb: установите переменную окружения GEMINI_KEY и запустите через Python.